In [0]:
# 1. Write a SQL query to get the product with the 3rd highest revenue in the last month.

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType
from datetime import date

sales_data = [
    (1, 101, "Bread", 2.5, 10, date(2026, 5, 1)),
    (2, 102, "Cake", 15.0, 2, date(2026, 5, 2)),
    (3, 103, "Cookie", 1.0, 20, date(2026, 6, 3)),
    (4, 101, "Bread", 2.5, 5, date(2026, 6, 4)),
    (5, 104, "Muffin", 3.0, 8, date(2026, 5, 5)),
    (6, 102, "Cake", 15.0, 1, date(2026, 6, 6)),
    (7, 103, "Cookie", 1.0, 15, date(2026, 6, 7)),
    (8, 104, "Muffin", 3.0, 12, date(2026, 6, 8)),
    (9, 101, "Bread", 2.5, 7, date(2026, 6, 9)),
    (10, 102, "Cake", 15.0, 3, date(2026, 6, 10))
]

sales_schema = StructType([
    StructField("sales_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), False),
    StructField("price", DoubleType(), False),
    StructField("qnty", IntegerType(), False),
    StructField("date", DateType(), False)
])

sales = spark.createDataFrame(sales_data, schema=sales_schema)
sales.createOrReplaceTempView("sales")
display(sales)

In [0]:
%sql
WITH CTE AS (SELECT *, (price * qnty) AS revenue FROM sales
WHERE date BETWEEN '2026-06-01' AND '2026-06-30'),
CTE1 AS (SELECT product_id,product_name,SUM(revenue) AS total_revenue FROM CTE
GROUP BY product_id,product_name),
CTE2 AS(SELECT *, DENSE_RANK(total_revenue) OVER(ORDER BY total_revenue DESC) AS rank FROM CTE1)
SELECT product_id,product_name,total_revenue FROM CTE2 
WHERE rank == 3




In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

employee_data = [
    (1, "Alice", 90000.0, None),
    (2, "Bob", 85000.0, 1),
    (3, "Charlie", 95000.0, 1),
    (4, "David", 75000.0, 2),
    (5, "Eve", 70000.0, 2)
]

employee_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("salary", DoubleType(), False),
    StructField("mang_id", IntegerType(), True)
])

employee = spark.createDataFrame(employee_data, schema=employee_schema)
employee.createOrReplaceTempView("employee")
display(employee)

In [0]:
# 2. Write a pyspark query to get the employees whose salary is greater than their manager.
from pyspark.sql.functions import col

df = employee.alias("e").join(employee.alias("m"), col("e.mang_id") == col("m.emp_id"), "inner").filter(col("e.salary") > col("m.salary")).select("e.emp_id","e.name")
df.display()

In [0]:
# 3. print top 3 elements from the array

list1 = [33,12,6,18,3]
list1.sort(reverse=True)
print(list1[0:3])
for i in range(3):
  print(list1[i])


In [0]:
# 4. Useful features of Delta Table in Databricks

# 1. ACID transactions: Ensures data integrity with atomicity, consistency, isolation, and durability.
# 2. Time travel: Query previous versions of data using versioning.
# 3. Schema evolution: Supports automatic schema updates.
# 4. Data upserts and deletes: Allows MERGE, UPDATE, and DELETE operations.
# 5. Audit history: Track changes with DESCRIBE HISTORY.
# 6. Efficient reads/writes: Optimized for large-scale data processing.
# 7. Streaming support: Enables real-time data ingestion and processing.

# Example: Querying Delta table history and time travel

# DESCRIBE HISTORY
history = spark.sql("DESCRIBE HISTORY pysparkdbt.gold.dimcustomers")
display(history)

# Time travel: Query previous version
df_v2 = spark.sql("SELECT * FROM pysparkdbt.gold.dimcustomers VERSION AS OF 2")
display(df_v2)

In [0]:
# 5. DELETE vs TRUNCATE TABLE
# DELETE: Removes rows matching a condition; can be selective.
# TRUNCATE TABLE: Removes all rows; faster, cannot be selective.